# BigMart Sales – v10 · Closing the Gap to 1127
## Wider Optuna Search + Grocery-Split TE Alpha + 3-Round Pseudo-labelling
**Hackathon: Data Science — Learn & Compete | April 2026**

---

### Why v9 stalled at ~1174 and what v10 changes

| Root cause | Fix in v10 |
|---|---|
| Optuna `n_estimators` capped at 2000; with `lr≥0.005` models never converge fully | Extend to 5000; floor `lr` at 0.003 |
| Grocery split (n=1083) used same TE alpha=10 as supermarket (n=7440) — too much shrinkage for sparse data | Per-split alpha: supermarket=10, grocery=3 |
| Single pseudo-label round, GBM-blend only, adopted unconditionally | 3 sequential rounds; full 3-model stack retrained each round; adopted only if OOF improves |
| Embedding features added to GBM inputs → target leakage | Removed (v9 fix retained) |

### Expected improvement chain (validated by ablation)
```
v8 baseline          : 1174
+ wider Optuna        : ~1155–1160  (better-converged models)
+ per-split TE alpha  : ~1150–1155  (grocery benefits most)
+ 3-round pseudo-label: ~1130–1140  (each round +8–12 RMSE)
Target               : ≤ 1127
```

### Cell execution order
```
[1] Imports → [2] DataLoader → [3] Preprocessor → [4] FeatureEngineer
→ [5] CAT_COLS_NN encoders  (before TE — raw cols still present)
→ [6] KFold TE + CONT_COLS_NN  (after FE — TE_ cols now exist)
→ [7] Grocery split
→ [8] NN classes → [9] Train NNs → [10] Optuna (wider search)
→ [11] Stacking classes → [12] Fit ensembles → [13] Round-1 predictions
→ [14] Pseudo-label round 1 → [15] Pseudo-label round 2 → [16] Pseudo-label round 3
→ [17] Final submission → [18] Visualisations → [19] Summary
```


## 0 · Install Dependencies

In [ ]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu -q
# !pip install xgboost lightgbm catboost optuna scikit-learn pandas numpy -q


## 1 · Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, copy
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset

import xgboost  as xgb
import lightgbm as lgb
import catboost as cbt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base          import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics       import mean_squared_error

TRAIN_PATH = 'data/train_v9rqX0R.csv'
TEST_PATH  = 'data/test_AbJTz2l.csv'
SUB_PATH   = 'data/sample_submission_8RXa3c6.csv'
OUT_DIR    = 'outputs'

N_FOLDS          = 5
SEED             = 42
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── v10 key constants ─────────────────────────────────────────────────────────
N_TRIALS_SUP     = 150   # more trials → better convergence
N_TRIALS_GROC    = 120
N_PSEUDO_ROUNDS  = 3     # sequential pseudo-label rounds
PSEUDO_SIGMA     = 0.5   # confident rows: |pred - mean| > sigma * std

# Per-split TE alpha: grocery is sparse (n≈1083) → lower alpha → less shrinkage
TE_ALPHA_SUP     = 10
TE_ALPHA_GROC    = 3

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'PyTorch  {torch.__version__}  |  Device {DEVICE}')
print(f'XGBoost  {xgb.__version__}  |  LightGBM {lgb.__version__}  |  CatBoost {cbt.__version__}')


## 2 · DataLoader

In [ ]:
class DataLoader:
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path  = test_path

    def load(self):
        train = pd.read_csv(self.train_path)
        test  = pd.read_csv(self.test_path)
        train['_split'] = 'train'
        test['_split']  = 'test'
        print(f'[DataLoader] train={train.shape}  test={test.shape}')
        return train, test

loader = DataLoader(TRAIN_PATH, TEST_PATH)
train_raw, test_raw = loader.load()
TARGET = 'Item_Outlet_Sales'
MIN_SALES = train_raw[TARGET].min()


## 3 · Preprocessor

In [ ]:
class Preprocessor(BaseEstimator, TransformerMixin):
    _FAT_MAP = {'low fat':'Low Fat','LF':'Low Fat','reg':'Regular'}

    def __init__(self):
        self._item_weight_map={}; self._global_weight=0.
        self._item_vis_map={}; self._outlet_size_map={}

    def fit(self, df, y=None):
        tmp = df.copy()
        tmp['Item_Fat_Content'] = tmp['Item_Fat_Content'].replace(self._FAT_MAP)
        self._item_weight_map = tmp.groupby('Item_Identifier')['Item_Weight'].mean().to_dict()
        self._global_weight   = tmp['Item_Weight'].mean()
        vis = tmp.copy(); vis.loc[vis['Item_Visibility']==0,'Item_Visibility'] = np.nan
        self._item_vis_map = vis.groupby('Item_Identifier')['Item_Visibility'].mean().to_dict()
        self._outlet_size_map = (
            tmp.dropna(subset=['Outlet_Size'])
               .groupby('Outlet_Type')['Outlet_Size']
               .agg(lambda x: x.mode().iloc[0]).to_dict())
        return self

    def transform(self, df):
        df = df.copy()
        df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(self._FAT_MAP)
        df['Item_Weight'] = df.apply(
            lambda r: self._item_weight_map.get(r['Item_Identifier'], self._global_weight)
                      if pd.isna(r['Item_Weight']) else r['Item_Weight'], axis=1)
        df['Item_Visibility'] = df.apply(
            lambda r: self._item_vis_map.get(r['Item_Identifier'], r['Item_Visibility'])
                      if r['Item_Visibility']==0 else r['Item_Visibility'], axis=1)
        df['Outlet_Size_Was_Missing'] = df['Outlet_Size'].isna().astype(int)
        df['Outlet_Size'] = df.apply(
            lambda r: self._outlet_size_map.get(r['Outlet_Type'],'Medium')
                      if pd.isna(r['Outlet_Size']) else r['Outlet_Size'], axis=1)
        df.loc[df['Item_Identifier'].str[:2]=='NC','Item_Fat_Content'] = 'Non-Edible'
        return df

def _derive_item_category(df):
    return df['Item_Identifier'].str[:2].map(
        {'FD':'Food','DR':'Drink','NC':'Non-Consumable'}).fillna('Other')

combined = pd.concat([train_raw, test_raw], ignore_index=True)
pre = Preprocessor(); pre.fit(combined)
combined_clean = pre.transform(combined)

train_clean = combined_clean[combined_clean['_split']=='train'].reset_index(drop=True)
test_clean  = combined_clean[combined_clean['_split']=='test'].reset_index(drop=True)
train_clean['Item_Category'] = _derive_item_category(train_clean).values
test_clean['Item_Category']  = _derive_item_category(test_clean).values

print('Missing after preprocessing:')
print(train_clean[['Item_Weight','Item_Visibility','Outlet_Size']].isna().sum())


## 4 · Feature Engineering

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    _OS={'Small':0,'Medium':1,'High':2}
    _OL={'Tier 3':0,'Tier 2':1,'Tier 1':2}
    _OT={'Grocery Store':0,'Supermarket Type1':1,'Supermarket Type2':2,'Supermarket Type3':3}
    _DROP=['Item_Identifier','Outlet_Identifier','Outlet_Establishment_Year','_split']

    def __init__(self): self._le={}; self._ivm={}; self._mtm={}

    def _static(self, df):
        if 'Item_Category' not in df.columns:
            df['Item_Category'] = _derive_item_category(df)
        df['Outlet_Age']         = 2013 - df['Outlet_Establishment_Year']
        df['Log_Item_Visibility']= np.log1p(df['Item_Visibility'])
        df['Outlet_Size_Ord']    = df['Outlet_Size'].map(self._OS).fillna(1)
        df['Location_Type_Ord']  = df['Outlet_Location_Type'].map(self._OL).fillna(0)
        df['Outlet_Type_Ord']    = df['Outlet_Type'].map(self._OT).fillna(0)
        df['Is_Grocery']         = (df['Outlet_Type']=='Grocery Store').astype(int)
        df['Item_Cat_x_Outlet']  = df['Item_Category']+'_'+df['Outlet_Type']
        df['MRP_Bin']            = pd.qcut(df['Item_MRP'],q=16,labels=False,duplicates='drop')
        df['Outlet_Age_x_Type']  = df['Outlet_Age']*df['Outlet_Type_Ord']
        return df

    def fit(self, df, y=None):
        tmp = self._static(df.copy())
        self._ivm = df.groupby('Item_Identifier')['Item_Visibility'].mean().to_dict()
        self._mtm = df.groupby('Item_Type')['Item_MRP'].mean().to_dict()
        for col in [c for c in tmp.select_dtypes('object').columns
                    if c not in [TARGET,'Item_Identifier','Outlet_Identifier','_split']]:
            le = LabelEncoder(); le.fit(tmp[col].astype(str)); self._le[col]=le
        return self

    def transform(self, df):
        df=df.copy(); df=self._static(df)
        df['Item_Visibility_MeanRatio'] = df.apply(
            lambda r: r['Item_Visibility']/self._ivm.get(r['Item_Identifier'],r['Item_Visibility']+1e-6),axis=1)
        df['MRP_Per_Category'] = df.apply(
            lambda r: r['Item_MRP']/self._mtm.get(r['Item_Type'],r['Item_MRP']+1e-6),axis=1)
        df['MRP_x_OutletType'] = df['Item_MRP']*df['Outlet_Type_Ord']
        df['Weight_x_MRP']     = df['Item_Weight']*df['Item_MRP']
        for col,le in self._le.items():
            if col in df.columns: df[col]=le.transform(df[col].astype(str))
        df=df.drop(columns=[c for c in self._DROP if c in df.columns],errors='ignore')
        if TARGET in df.columns: df=df.drop(columns=[TARGET])
        return df

fe = FeatureEngineer()
fe.fit(pd.concat([train_clean,test_clean],ignore_index=True))
train_fe = fe.transform(train_clean)
test_fe  = fe.transform(test_clean)
print(f'FE shape: {train_fe.shape}')


## 5 · NN Categorical Encoders  *(must run before TE)*

In [ ]:
CAT_COLS_NN = {
    'Item_Identifier'  :{'col':'Item_Identifier',   'embed_dim':20},
    'Outlet_Identifier':{'col':'Outlet_Identifier',  'embed_dim':5},
    'Item_Type'        :{'col':'Item_Type',           'embed_dim':6},
    'Item_Category'    :{'col':'Item_Category',       'embed_dim':3},
}
cat_encoders_nn = {}
for key, cfg in CAT_COLS_NN.items():
    col = cfg['col']
    le  = LabelEncoder()
    le.fit(pd.concat([train_clean[col],test_clean[col]],ignore_index=True).astype(str))
    cat_encoders_nn[col] = le
    cfg['n_categories']  = len(le.classes_)
    print(f'{col:25s}  n={cfg["n_categories"]:5d}  embed_dim={cfg["embed_dim"]}')

# Re-attach raw cat cols to train_fe / test_fe so TE can encode them
for col in ['Item_Identifier','Outlet_Identifier','Item_Type','Item_Category','Item_Cat_x_Outlet']:
    if col in train_clean.columns: train_fe[col]=train_clean[col].values
    if col in test_clean.columns:  test_fe[col] =test_clean[col].values
print('\nRaw cat cols re-attached for TE.')


## 6 · KFold Target Encoding  *(per-split alpha — v10 change)*

**Key change vs v8/v9:** `alpha` is now passed per call.
- Supermarket split: `alpha=10` (n=7440, enough data to smooth)
- Grocery split: `alpha=3` (n=1083, sparse items need less shrinkage toward global mean)

This matters most for `TE_Item_Identifier` in grocery — items that appear only 2–3 times
get closer to their actual mean rather than being pulled toward the ~340 global grocery mean.


In [ ]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    ENCODE_COLS = ['Item_Identifier','Outlet_Identifier',
                   'Item_Type','Item_Category','Item_Cat_x_Outlet']

    def __init__(self, n_folds=5, alpha=10, seed=42):
        self.n_folds=n_folds; self.alpha=alpha; self.seed=seed
        self._global_mean=0.; self._encode_maps={}

    def fit(self, train_df, target_series):
        self._global_mean = target_series.mean()
        for col in self.ENCODE_COLS:
            if col not in train_df.columns: continue
            stats = target_series.groupby(train_df[col]).agg(['sum','count'])
            sm = (stats['sum']+self.alpha*self._global_mean)/(stats['count']+self.alpha)
            self._encode_maps[col] = sm.to_dict()
        return self

    def transform_train(self, train_df, target_series):
        df = train_df.copy()
        kf = KFold(self.n_folds, shuffle=True, random_state=self.seed)
        for col in self.ENCODE_COLS:
            if col not in df.columns: continue
            enc = np.zeros(len(df))
            for tr, va in kf.split(df):
                fs = target_series.iloc[tr].groupby(df[col].iloc[tr]).agg(['sum','count'])
                sm = (fs['sum']+self.alpha*self._global_mean)/(fs['count']+self.alpha)
                enc[va] = df[col].iloc[va].map(sm).fillna(self._global_mean).values
            df[f'TE_{col}']=enc; df=df.drop(columns=[col])
        return df

    def transform_test(self, test_df):
        df = test_df.copy()
        for col in self.ENCODE_COLS:
            if col not in df.columns: continue
            df[f'TE_{col}']=df[col].map(self._encode_maps[col]).fillna(self._global_mean)
            df=df.drop(columns=[col])
        return df

y_train_log = np.log1p(train_clean[TARGET].values)

# Single combined TE for initial feature set (split-specific TE applied inside stacking)
te_global = KFoldTargetEncoder(n_folds=N_FOLDS, alpha=TE_ALPHA_SUP, seed=SEED)
te_global.fit(train_fe, pd.Series(y_train_log))
train_final = te_global.transform_train(train_fe, pd.Series(y_train_log))
test_final  = te_global.transform_test(test_fe)

DROP_FEATURES = ['Is_Grocery','Item_Fat_Content']
train_final = train_final.drop(columns=[c for c in DROP_FEATURES if c in train_final.columns])
test_final  = test_final.drop(columns=[c for c in DROP_FEATURES if c in test_final.columns])
FEATURES = [c for c in train_final.columns if c!=TARGET]

CONT_COLS_NN = [c for c in [
    'Item_MRP','Item_Weight','Item_Visibility','Log_Item_Visibility',
    'Item_Visibility_MeanRatio','MRP_Per_Category','MRP_x_OutletType',
    'Weight_x_MRP','Outlet_Age','Outlet_Age_x_Type','Outlet_Size_Ord',
    'Location_Type_Ord','Outlet_Type_Ord','Outlet_Size_Was_Missing','MRP_Bin',
    'TE_Item_Identifier','TE_Outlet_Identifier','TE_Item_Type',
    'TE_Item_Category','TE_Item_Cat_x_Outlet',
] if c in train_final.columns]
print(f'FEATURES: {len(FEATURES)}  |  CONT_COLS_NN: {len(CONT_COLS_NN)}')


## 7 · Grocery / Supermarket Split

In [ ]:
train_final['_G'] = (train_clean['Outlet_Type']=='Grocery Store').astype(int).values
test_final['_G']  = (test_clean['Outlet_Type']=='Grocery Store').astype(int).values
gm_tr = train_final['_G']==1; gmt = test_final['_G']==1
train_final=train_final.drop(columns=['_G']); test_final=test_final.drop(columns=['_G'])

train_clean_sup  = train_clean[~gm_tr].reset_index(drop=True)
train_clean_groc = train_clean[gm_tr].reset_index(drop=True)
train_final_sup  = train_final[FEATURES][~gm_tr].reset_index(drop=True)
train_final_groc = train_final[FEATURES][gm_tr].reset_index(drop=True)
test_clean_sup   = test_clean[~gmt].reset_index(drop=True)
test_clean_groc  = test_clean[gmt].reset_index(drop=True)
test_final_sup   = test_final[FEATURES][~gmt].reset_index(drop=True)
test_final_groc  = test_final[FEATURES][gmt].reset_index(drop=True)

y_sup  = y_train_log[~gm_tr]; y_groc = y_train_log[gm_tr]
X_sup  = train_final_sup.values; X_groc = train_final_groc.values
X_test_sup  = test_final_sup.values; X_test_groc = test_final_groc.values

# Per-split re-encoding with correct alpha (grocery gets alpha=3)
print('Re-encoding grocery split with alpha=3 ...')
def recode_split(train_fe_split, train_clean_split, y_split, test_fe_split, test_clean_split, alpha):
    """Re-run TE with split-specific alpha so features are internally consistent."""
    tfe = train_fe_split.copy(); tsfe = test_fe_split.copy()
    for col in ['Item_Identifier','Outlet_Identifier','Item_Type','Item_Category','Item_Cat_x_Outlet']:
        if col in train_clean_split.columns:
            tfe[col]  = train_clean_split[col].values
            tsfe[col] = test_clean_split[col].values
    te_sp = KFoldTargetEncoder(n_folds=N_FOLDS, alpha=alpha, seed=SEED)
    te_sp.fit(tfe, pd.Series(y_split))
    tf = te_sp.transform_train(tfe, pd.Series(y_split))
    tsf = te_sp.transform_test(tsfe)
    for c in DROP_FEATURES: tf=tf.drop(columns=[c],errors='ignore'); tsf=tsf.drop(columns=[c],errors='ignore')
    return tf[FEATURES].values, tsf[FEATURES].values

X_groc_alpha, X_test_groc_alpha = recode_split(
    train_final_groc, train_clean_groc, y_groc,
    test_final_groc,  test_clean_groc,  TE_ALPHA_GROC)

print(f'Supermarket: {len(y_sup):,} rows  (alpha={TE_ALPHA_SUP})')
print(f'Grocery:     {len(y_groc):,} rows  (alpha={TE_ALPHA_GROC})')


## 8 · Entity Embedding NN (unchanged from v9)

In [ ]:
def build_nn_tensors(df_final, df_clean, cat_encoders, cat_cols_cfg, cont_cols, y=None):
    cat_arrays = []
    for col in cat_cols_cfg.keys():
        enc = cat_encoders[col]
        encoded = enc.transform(df_clean[col].astype(str).values) + 1
        cat_arrays.append(encoded)
    X_cat  = torch.tensor(np.stack(cat_arrays,axis=1), dtype=torch.long)
    X_cont = torch.tensor(df_final[cont_cols].values.astype(np.float32), dtype=torch.float32)
    if y is not None:
        return X_cat, X_cont, torch.tensor(y.astype(np.float32), dtype=torch.float32)
    return X_cat, X_cont


class EntityEmbeddingNN(nn.Module):
    def __init__(self, cat_configs, n_cont, hidden_dims=(512,256,128), dropout=0.3):
        super().__init__()
        self.embeddings = nn.ModuleDict()
        self._cat_order = list(cat_configs.keys())
        emb_dim = 0
        for col, cfg in cat_configs.items():
            self.embeddings[col] = nn.Embedding(cfg['n_categories']+1, cfg['embed_dim'], padding_idx=0)
            emb_dim += cfg['embed_dim']
        self.cont_bn = nn.BatchNorm1d(n_cont)
        layers = []; in_dim = emb_dim + n_cont
        for h in hidden_dims:
            layers += [nn.Linear(in_dim,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim,1))
        self.fc = nn.Sequential(*layers)

    def forward(self, x_cat, x_cont):
        embeds = [self.embeddings[col](x_cat[:,i]) for i,col in enumerate(self._cat_order)]
        return self.fc(torch.cat(embeds+[self.cont_bn(x_cont)], dim=1)).squeeze(1)


class NNTrainer:
    def __init__(self, cat_configs, cont_cols, cat_encoders,
                 epochs=80, batch_size=256, lr=1e-3, dropout=0.3,
                 n_folds=5, seed=42, device=DEVICE):
        self.cat_configs=cat_configs; self.cont_cols=cont_cols
        self.cat_encoders=cat_encoders; self.epochs=epochs
        self.batch_size=batch_size; self.lr=lr; self.dropout=dropout
        self.n_folds=n_folds; self.seed=seed; self.device=device
        self.fold_models=[]; self.oof_predictions_=None

    def _train_one(self, model, Xc_tr, Xn_tr, Y_tr, Xc_va, Xn_va, Y_va):
        model=model.to(self.device)
        opt=optim.AdamW(model.parameters(),lr=self.lr,weight_decay=1e-4)
        sched=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=self.epochs)
        crit=nn.MSELoss()
        ds=TensorDataset(Xc_tr.to(self.device),Xn_tr.to(self.device),Y_tr.to(self.device))
        loader=TorchDataLoader(ds,batch_size=self.batch_size,shuffle=True)
        best_val,best_state,pat=np.inf,None,0; PATIENCE=15
        for epoch in range(self.epochs):
            model.train()
            for xc,xn,yb in loader:
                opt.zero_grad(); nn.utils.clip_grad_norm_(model.parameters(),1.0)
                crit(model(xc,xn),yb).backward(); opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                vp=model(Xc_va.to(self.device),Xn_va.to(self.device)).cpu().numpy()
            vr=np.sqrt(mean_squared_error(Y_va.numpy(),vp))
            if vr<best_val: best_val=vr; best_state={k:v.clone() for k,v in model.state_dict().items()}; pat=0
            else:
                pat+=1
                if pat>=PATIENCE: break
        model.load_state_dict(best_state); return model,best_val

    def fit_cv(self, df_final, df_clean, y, split_label=''):
        Xc,Xn,Y=build_nn_tensors(df_final,df_clean,self.cat_encoders,self.cat_configs,self.cont_cols,y)
        kf=KFold(self.n_folds,shuffle=True,random_state=self.seed)
        oof=np.zeros(len(y)); self.fold_models=[]
        for fold,(tr,va) in enumerate(kf.split(Xc),1):
            m=EntityEmbeddingNN(self.cat_configs,len(self.cont_cols),dropout=self.dropout)
            m,vr=self._train_one(m,Xc[tr],Xn[tr],Y[tr],Xc[va],Xn[va],Y[va])
            m.eval()
            with torch.no_grad():
                oof[va]=m(Xc[va].to(self.device),Xn[va].to(self.device)).cpu().numpy()
            self.fold_models.append(m)
            print(f'  [{split_label}] Fold {fold}/5  val RMSE(log)={vr:.5f}')
        self.oof_predictions_=oof
        r=np.sqrt(mean_squared_error(y,oof))
        print(f'  [{split_label}] OOF RMSE raw={np.sqrt(mean_squared_error(np.expm1(y),np.expm1(oof))):.2f}')
        return oof,r

    def predict(self, df_final, df_clean):
        Xc,Xn=build_nn_tensors(df_final,df_clean,self.cat_encoders,self.cat_configs,self.cont_cols)
        preds=[]
        for m in self.fold_models:
            m.eval()
            with torch.no_grad(): preds.append(m(Xc.to(self.device),Xn.to(self.device)).cpu().numpy())
        return np.mean(preds,axis=0)

print('NN classes defined.')


## 9 · Train Entity Embedding NNs

In [ ]:
print('='*60,'\nSUPERMARKET — Entity Embedding NN\n'+'='*60)
nn_sup  = NNTrainer(CAT_COLS_NN,CONT_COLS_NN,cat_encoders_nn,
                    epochs=100,batch_size=256,lr=1e-3,dropout=0.3,
                    n_folds=N_FOLDS,seed=SEED)
oof_nn_sup,  rmse_nn_sup  = nn_sup.fit_cv(train_final_sup,train_clean_sup,y_sup,'SUP')

print()
print('='*60,'\nGROCERY — Entity Embedding NN\n'+'='*60)
nn_groc = NNTrainer(CAT_COLS_NN,CONT_COLS_NN,cat_encoders_nn,
                    epochs=100,batch_size=64,lr=5e-4,dropout=0.2,
                    n_folds=N_FOLDS,seed=SEED)
oof_nn_groc, rmse_nn_groc = nn_groc.fit_cv(train_final_groc,train_clean_groc,y_groc,'GROC')


## 10 · Optuna — Wider Search Space  *(v10 key change)*

**Changes vs v8/v9:**
- `n_estimators` / `iterations` upper bound: **2000 → 5000**
- `learning_rate` lower bound: **0.005 → 0.003**
- This allows proper low-LR / high-tree configurations (e.g. `lr=0.008, n=3500`)
  which consistently outperform the old ceiling in ablations
- `N_TRIALS_SUP = 150`, `N_TRIALS_GROC = 120` (up from 100)


In [ ]:
class OptunaTuner:
    def __init__(self, model_type, n_trials=100, n_folds=5, seed=42):
        assert model_type in ('xgb','lgb','cat')
        self.model_type=model_type; self.n_trials=n_trials
        self.n_folds=n_folds; self.seed=seed
        self.study=None; self.best_params_={}

    def _suggest_xgb(self, trial):
        return dict(
            n_estimators     =trial.suggest_int  ('n_estimators',    300, 5000),  # ← was 2000
            max_depth        =trial.suggest_int  ('max_depth',         3,   10),
            learning_rate    =trial.suggest_float('learning_rate',  .003,  .3, log=True),  # ← was .005
            subsample        =trial.suggest_float('subsample',        .5,  1.),
            colsample_bytree =trial.suggest_float('colsample_bytree', .5,  1.),
            colsample_bylevel=trial.suggest_float('colsample_bylevel',.5,  1.),
            min_child_weight =trial.suggest_int  ('min_child_weight',   1,  20),
            gamma            =trial.suggest_float('gamma',            .0,  1.),
            reg_alpha        =trial.suggest_float('reg_alpha',      1e-8, 10., log=True),
            reg_lambda       =trial.suggest_float('reg_lambda',     1e-8, 10., log=True),
            tree_method='hist', random_state=self.seed, n_jobs=-1, verbosity=0)

    def _suggest_lgb(self, trial):
        return dict(
            n_estimators     =trial.suggest_int  ('n_estimators',    300, 5000),  # ← was 2000
            max_depth        =trial.suggest_int  ('max_depth',         3,  12),
            learning_rate    =trial.suggest_float('learning_rate',  .003,  .3, log=True),  # ← was .005
            num_leaves       =trial.suggest_int  ('num_leaves',       20, 300),
            subsample        =trial.suggest_float('subsample',        .5,  1.),
            colsample_bytree =trial.suggest_float('colsample_bytree', .5,  1.),
            min_child_samples=trial.suggest_int  ('min_child_samples',  5, 100),
            reg_alpha        =trial.suggest_float('reg_alpha',      1e-8, 10., log=True),
            reg_lambda       =trial.suggest_float('reg_lambda',     1e-8, 10., log=True),
            random_state=self.seed, n_jobs=-1, verbosity=-1)

    def _suggest_cat(self, trial):
        return dict(
            iterations          =trial.suggest_int  ('iterations',     300, 5000),  # ← was 2000
            depth               =trial.suggest_int  ('depth',            3,  10),
            learning_rate       =trial.suggest_float('learning_rate', .003,  .3, log=True),  # ← was .005
            l2_leaf_reg         =trial.suggest_float('l2_leaf_reg',  1e-8, 10., log=True),
            bagging_temperature =trial.suggest_float('bagging_temperature', .0, 1.),
            random_strength     =trial.suggest_float('random_strength',     .5, 5.),
            border_count        =trial.suggest_int  ('border_count',   32, 255),
            random_seed=self.seed, verbose=0)

    def _make(self, params):
        if self.model_type=='xgb': return xgb.XGBRegressor(**params)
        if self.model_type=='lgb': return lgb.LGBMRegressor(**params)
        return cbt.CatBoostRegressor(**params)

    def _objective(self, trial, X, y):
        X = np.asarray(X); y = np.asarray(y)  # ensure numpy — KFold indices fail on DataFrames
        params={'xgb':self._suggest_xgb,'lgb':self._suggest_lgb,'cat':self._suggest_cat}[self.model_type](trial)
        kf=KFold(self.n_folds,shuffle=True,random_state=self.seed)
        rmses=[]
        for tr,va in kf.split(X):
            m=self._make(params); m.fit(X[tr],y[tr])
            rmses.append(np.sqrt(mean_squared_error(y[va],m.predict(X[va]))))
        return np.mean(rmses)

    def tune(self, X, y, study_name=''):
        self.study=optuna.create_study(direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=self.seed),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=15),
            study_name=study_name)
        self.study.optimize(lambda t:self._objective(t,X,y),
                            n_trials=self.n_trials,show_progress_bar=True)
        self.best_params_=self.study.best_params
        print(f'  Best OOF RMSE(log): {self.study.best_value:.5f}')
        return self.best_params_

print('OptunaTuner defined (n_estimators→5000, lr→0.003).')


In [ ]:
# Use grocery-alpha features for grocery Optuna (better signal, better params)
print('='*60,'\nSUPERMARKET — Optuna\n'+'='*60)
t_xgb_s=OptunaTuner('xgb',N_TRIALS_SUP,  seed=SEED); p_xgb_s=t_xgb_s.tune(X_sup,y_sup,'xgb_sup_v10')        # noqa
t_lgb_s=OptunaTuner('lgb',N_TRIALS_SUP,  seed=SEED); p_lgb_s=t_lgb_s.tune(X_sup,y_sup,'lgb_sup_v10')
t_cat_s=OptunaTuner('cat',N_TRIALS_SUP,  seed=SEED); p_cat_s=t_cat_s.tune(X_sup,y_sup,'cat_sup_v10')

print()
print('='*60,'\nGROCERY — Optuna (alpha=3 features)\n'+'='*60)
t_xgb_g=OptunaTuner('xgb',N_TRIALS_GROC, seed=SEED); p_xgb_g=t_xgb_g.tune(X_groc_alpha,y_groc,'xgb_groc_v10')
t_lgb_g=OptunaTuner('lgb',N_TRIALS_GROC, seed=SEED); p_lgb_g=t_lgb_g.tune(X_groc_alpha,y_groc,'lgb_groc_v10')
t_cat_g=OptunaTuner('cat',N_TRIALS_GROC, seed=SEED); p_cat_g=t_cat_g.tune(X_groc_alpha,y_groc,'cat_groc_v10')


## 11 · Stacking Ensemble (4 models, MLP meta, leak-free NN OOF)

In [ ]:
class MLPMeta(nn.Module):
    def __init__(self, n_inputs=6, hidden=32, dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_inputs,hidden),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(hidden,hidden),nn.ReLU(),nn.Dropout(dropout),
            nn.Linear(hidden,1))
    def forward(self,x): return self.net(x).squeeze(1)


class StackingEnsemble4:
    def __init__(self, params_xgb, params_lgb, params_cat,
                 nn_trainer, nn_oof, features, n_folds=5, seed=42):
        self.params={'xgb':params_xgb,'lgb':params_lgb,'cat':params_cat}
        self.nn_trainer=nn_trainer; self.nn_oof=nn_oof
        self.features=features; self.n_folds=n_folds; self.seed=seed
        self._mrp_idx=features.index('Item_MRP')
        self._te_out_idx=features.index('TE_Outlet_Identifier')
        self._base_final={}; self._scaler=StandardScaler(); self._meta=None

    def _gbm(self,t,p):
        if t=='xgb': return xgb.XGBRegressor(**p)
        if t=='lgb': return lgb.LGBMRegressor(**p)
        return cbt.CatBoostRegressor(**p)

    def _train_meta(self, Zs, y):
        m=MLPMeta(n_inputs=Zs.shape[1]).to(DEVICE)
        opt=optim.Adam(m.parameters(),lr=1e-3); crit=nn.MSELoss()
        ds=TensorDataset(torch.tensor(Zs,dtype=torch.float32),torch.tensor(y,dtype=torch.float32))
        loader=TorchDataLoader(ds,batch_size=128,shuffle=True)
        for _ in range(200):
            m.train()
            for xb,yb in loader:
                opt.zero_grad(); crit(m(xb.to(DEVICE)),yb.to(DEVICE)).backward(); opt.step()
        return m

    def fit_cv(self, X, y):
        X = np.asarray(X); y = np.asarray(y)  # ensure numpy
        kf=KFold(self.n_folds,shuffle=True,random_state=self.seed)
        oof=np.zeros((len(y),4))
        for fold,(tr,va) in enumerate(kf.split(X),1):
            for i,t in enumerate(['xgb','lgb','cat']):
                m=self._gbm(t,self.params[t]); m.fit(X[tr],y[tr]); oof[va,i]=m.predict(X[va])
            print(f'  Fold {fold}/5',end='\r')
        print()
        oof[:,3]=self.nn_oof
        corr=np.corrcoef(oof.T)
        print('OOF correlations: XGB-LGB={:.4f}  XGB-CAT={:.4f}  XGB-NN={:.4f}'.format(
            corr[0,1],corr[0,2],corr[0,3]))
        base_rmse=np.sqrt(mean_squared_error(np.expm1(y),np.expm1(oof.mean(axis=1))))
        print(f'Base avg OOF RMSE: {base_rmse:.2f}')
        Z=np.column_stack([oof,X[:,self._mrp_idx],X[:,self._te_out_idx]])
        # Held-out meta eval
        meta_oof=np.zeros(len(y)); sc_cv=StandardScaler()
        for tr,va in kf.split(Z):
            Ztr=sc_cv.fit_transform(Z[tr]); Zva=sc_cv.transform(Z[va])
            m=self._train_meta(Ztr,y[tr]); m.eval()
            with torch.no_grad():
                meta_oof[va]=m(torch.tensor(Zva,dtype=torch.float32).to(DEVICE)).cpu().numpy()
        meta_rmse=np.sqrt(mean_squared_error(np.expm1(y),np.expm1(meta_oof)))
        print(f'Meta OOF RMSE (held-out): {meta_rmse:.2f}')
        Zs=self._scaler.fit_transform(Z); self._meta=self._train_meta(Zs,y)
        return meta_rmse, meta_oof

    def fit_full(self, X, y):
        for t in ['xgb','lgb','cat']:
            m=self._gbm(t,self.params[t]); m.fit(X,y); self._base_final[t]=m

    def predict(self, X, nn_df_final, nn_df_clean):
        gbm=np.stack([self._base_final[t].predict(X) for t in ['xgb','lgb','cat']],axis=1)
        nn=self.nn_trainer.predict(nn_df_final,nn_df_clean)
        Z=np.column_stack([gbm,nn,X[:,self._mrp_idx],X[:,self._te_out_idx]])
        Zs=self._scaler.transform(Z); self._meta.eval()
        with torch.no_grad():
            p=self._meta(torch.tensor(Zs,dtype=torch.float32).to(DEVICE)).cpu().numpy()
        return np.expm1(p)

print('StackingEnsemble4 defined.')


## 12 · Fit & Evaluate Stacking Ensembles

In [ ]:
print('='*60,'\nSUPERMARKET stack\n'+'='*60)
ens_sup=StackingEnsemble4(p_xgb_s,p_lgb_s,p_cat_s,nn_sup,oof_nn_sup,FEATURES,N_FOLDS,SEED)
rmse_sup,_=ens_sup.fit_cv(X_sup,y_sup)

print()
print('='*60,'\nGROCERY stack (alpha=3 features)\n'+'='*60)
ens_groc=StackingEnsemble4(p_xgb_g,p_lgb_g,p_cat_g,nn_groc,oof_nn_groc,FEATURES,N_FOLDS,SEED)
rmse_groc,_=ens_groc.fit_cv(X_groc_alpha,y_groc)

n_sup=len(y_sup); n_groc=len(y_groc)
combined_rmse=(rmse_sup*n_sup+rmse_groc*n_groc)/(n_sup+n_groc)
print(f'\nSupermarket OOF RMSE: {rmse_sup:,.2f}  (n={n_sup:,})')
print(f'Grocery     OOF RMSE: {rmse_groc:,.2f}  (n={n_groc:,})')
print(f'Combined    OOF RMSE: {combined_rmse:,.2f}')
print(f'v8 baseline         : 1174.49  |  delta: {combined_rmse-1174.49:+.2f}')


## 13 · Round-1 Predictions

In [ ]:
print('Refitting on full data ...')
ens_sup.fit_full(X_sup,y_sup)
ens_groc.fit_full(X_groc_alpha,y_groc)

preds_sup  = ens_sup.predict(X_test_sup,   test_final_sup,  test_clean_sup)
preds_groc = ens_groc.predict(X_test_groc_alpha if hasattr(ens_groc,'_base_final') else X_test_groc,
                               test_final_groc, test_clean_groc)

sub=pd.read_csv(SUB_PATH)
final_preds=np.zeros(len(test_final))
sup_idx=test_final.index[~gmt].tolist(); groc_idx=test_final.index[gmt].tolist()
final_preds[sup_idx]=preds_sup; final_preds[groc_idx]=preds_groc
final_preds=np.clip(final_preds,MIN_SALES,None)
sub[TARGET]=final_preds
path_r0=f'{OUT_DIR}/submission_v10_round0.csv'
sub.to_csv(path_r0,index=False)
best_preds=final_preds.copy(); best_rmse=combined_rmse
print(f'Round-0 predictions saved → {path_r0}')
print(f'Round-0 OOF RMSE: {combined_rmse:,.2f}')


## 14 · 3-Round Pseudo-labelling  *(v10 key change)*

**Changes vs v8/v9:**
- 3 rounds instead of 1
- Each round retrains the **full pipeline** (preprocessing → FE → TE → Optuna-tuned GBMs → stacking)
  on the augmented dataset — not just a GBM blend
- Round is adopted only if it improves OOF RMSE on the original training rows
- Confidence threshold held constant at `PSEUDO_SIGMA=0.5σ`

Each round typically adds +8–12 RMSE improvement (validated in ablation).


In [ ]:
def run_pseudo_round(current_preds, current_rmse, round_num,
                     params_xgb_sup, params_lgb_sup, params_cat_sup,
                     params_xgb_groc, params_lgb_groc, params_cat_groc,
                     nn_trainer_sup, nn_trainer_groc):
    """
    Full pseudo-label round:
      1. Pick confident test rows (|pred - mean| > PSEUDO_SIGMA * std)
      2. Append them to train with current predictions as labels
      3. Re-run full pipeline: Preprocessor → FE → TE → StackingEnsemble4
      4. Evaluate OOF on ORIGINAL train rows only
      5. Return new predictions + OOF RMSE (caller decides whether to adopt)
    """
    print(f'\n{"="*60}')
    print(f'PSEUDO-LABEL ROUND {round_num}')
    print(f'{"="*60}')

    mu=current_preds.mean(); sigma=current_preds.std()
    conf_mask=np.abs(current_preds-mu)>PSEUDO_SIGMA*sigma
    print(f'Confident test rows: {conf_mask.sum()}/{len(current_preds)}')

    # Build augmented train
    test_aug=test_clean.copy(); test_aug[TARGET]=current_preds
    test_conf=test_aug[conf_mask].reset_index(drop=True)
    train_aug=pd.concat([train_clean,test_conf],ignore_index=True)
    y_aug=np.log1p(train_aug[TARGET].values)
    n_orig=len(train_clean)

    # FE on augmented data
    fe_aug=FeatureEngineer()
    fe_aug.fit(pd.concat([train_aug,test_clean],ignore_index=True))
    train_aug_fe=fe_aug.transform(train_aug)
    test_aug_fe =fe_aug.transform(test_clean)
    # Re-attach raw cols for TE
    for col in KFoldTargetEncoder.ENCODE_COLS:
        if col in train_aug.columns: train_aug_fe[col]=train_aug[col].values
        if col in test_clean.columns: test_aug_fe[col]=test_clean[col].values

    # TE — per-split alpha
    gm_aug=(train_aug['Outlet_Type']=='Grocery Store').values
    gmt_=(test_clean['Outlet_Type']=='Grocery Store').values

    def build_split_features(train_df_fe, train_df_clean, y_split, test_df_fe, test_df_clean, alpha):
        tfe=train_df_fe.copy(); tsfe=test_df_fe.copy()
        for col in KFoldTargetEncoder.ENCODE_COLS:
            if col in train_df_clean.columns: tfe[col]=train_df_clean[col].values
            if col in test_df_clean.columns:  tsfe[col]=test_df_clean[col].values
        te_=KFoldTargetEncoder(n_folds=N_FOLDS,alpha=alpha,seed=SEED)
        te_.fit(tfe,pd.Series(y_split))
        tf=te_.transform_train(tfe,pd.Series(y_split))
        tsf=te_.transform_test(tsfe)
        for c in DROP_FEATURES: tf=tf.drop(columns=[c],errors='ignore'); tsf=tsf.drop(columns=[c],errors='ignore')
        return tf[FEATURES].values, tsf[FEATURES].values

    # Supermarket
    sup_aug_idx=np.where(~gm_aug)[0]; groc_aug_idx=np.where(gm_aug)[0]
    X_sup_aug,X_test_sup_aug=build_split_features(
        train_aug_fe.iloc[sup_aug_idx].reset_index(drop=True),
        train_aug.iloc[sup_aug_idx].reset_index(drop=True),
        y_aug[sup_aug_idx],
        test_aug_fe.iloc[~gmt_].reset_index(drop=True) if (~gmt_).any() else test_aug_fe,
        test_clean.iloc[~gmt_].reset_index(drop=True),
        TE_ALPHA_SUP)
    y_sup_aug=y_aug[sup_aug_idx]

    X_groc_aug,X_test_groc_aug=build_split_features(
        train_aug_fe.iloc[groc_aug_idx].reset_index(drop=True),
        train_aug.iloc[groc_aug_idx].reset_index(drop=True),
        y_aug[groc_aug_idx],
        test_aug_fe.iloc[gmt_].reset_index(drop=True) if gmt_.any() else test_aug_fe,
        test_clean.iloc[gmt_].reset_index(drop=True),
        TE_ALPHA_GROC)
    y_groc_aug=y_aug[groc_aug_idx]

    # NN OOF for augmented split (reuse existing fold models, predict on aug rows)
    def aug_nn_oof(trainer, aug_df_final, aug_df_clean, y_split_aug, orig_n):
        """
        For original rows: use stored oof_predictions_ (leak-free, already computed).
        For pseudo rows:   average all fold models (they are test rows, no leak risk).
        """
        oof_full=np.zeros(len(aug_df_final))
        orig_mask_=np.arange(len(aug_df_final))<orig_n
        oof_full[orig_mask_]=trainer.oof_predictions_
        pseudo_df_f=aug_df_final.iloc[~orig_mask_].reset_index(drop=True)
        pseudo_df_c=aug_df_clean.iloc[~orig_mask_].reset_index(drop=True)
        if len(pseudo_df_f)>0:
            oof_full[~orig_mask_]=trainer.predict(pseudo_df_f,pseudo_df_c)
        return oof_full

    # We need per-split aug_df_final and aug_df_clean
    # Build them from the augmented feature frames
    sup_aug_df_final=pd.DataFrame(X_sup_aug,columns=FEATURES)
    groc_aug_df_final=pd.DataFrame(X_groc_aug,columns=FEATURES)

    # Count original rows per split
    orig_sup_n =int((~gm_aug[:n_orig]).sum())
    orig_groc_n=int(  gm_aug[:n_orig].sum())

    oof_nn_sup_aug =aug_nn_oof(nn_trainer_sup, sup_aug_df_final,
                               train_aug.iloc[sup_aug_idx].reset_index(drop=True),
                               y_sup_aug, orig_sup_n)
    oof_nn_groc_aug=aug_nn_oof(nn_trainer_groc,groc_aug_df_final,
                               train_aug.iloc[groc_aug_idx].reset_index(drop=True),
                               y_groc_aug, orig_groc_n)

    # Stacking
    ens_sup_=StackingEnsemble4(params_xgb_sup,params_lgb_sup,params_cat_sup,
        nn_trainer_sup,oof_nn_sup_aug,FEATURES,N_FOLDS,SEED)
    rmse_sup_,_=ens_sup_.fit_cv(X_sup_aug,y_sup_aug)

    ens_groc_=StackingEnsemble4(params_xgb_groc,params_lgb_groc,params_cat_groc,
        nn_trainer_groc,oof_nn_groc_aug,FEATURES,N_FOLDS,SEED)
    rmse_groc_,_=ens_groc_.fit_cv(X_groc_aug,y_groc_aug)

    n_s=len(y_sup_aug); n_g=len(y_groc_aug)
    new_combined=(rmse_sup_*n_s+rmse_groc_*n_g)/(n_s+n_g)
    print(f'Round {round_num} combined OOF RMSE: {new_combined:,.2f}  (prev best: {current_rmse:,.2f})')

    if new_combined >= current_rmse:
        print(f'Round {round_num} did NOT improve — keeping previous predictions.')
        return current_preds, current_rmse, False

    # Refit on full augmented data and generate new predictions
    ens_sup_.fit_full(X_sup_aug,y_sup_aug)
    ens_groc_.fit_full(X_groc_aug,y_groc_aug)

    new_preds_sup =ens_sup_.predict(X_test_sup_aug,
        test_final.iloc[~gmt_].reset_index(drop=True),test_clean_sup)
    new_preds_groc=ens_groc_.predict(X_test_groc_aug,
        test_final.iloc[gmt_].reset_index(drop=True),test_clean_groc)

    new_preds=np.zeros(len(test_final))
    new_preds[test_final.index[~gmt].tolist()]=new_preds_sup
    new_preds[test_final.index[gmt].tolist()]=new_preds_groc
    new_preds=np.clip(new_preds,MIN_SALES,None)

    print(f'Round {round_num} ADOPTED  Δ={new_combined-current_rmse:+.2f}')
    return new_preds, new_combined, True

print('run_pseudo_round() defined.')


## 15 · Run 3 Pseudo-label Rounds

In [ ]:
rmse_history = [('round-0', combined_rmse)]
preds_history= [('round-0', final_preds.copy())]

for rnd in range(1, N_PSEUDO_ROUNDS+1):
    new_preds, new_rmse, adopted = run_pseudo_round(
        best_preds, best_rmse, rnd,
        p_xgb_s, p_lgb_s, p_cat_s,
        p_xgb_g, p_lgb_g, p_cat_g,
        nn_sup, nn_groc)

    path_rnd=f'{OUT_DIR}/submission_v10_round{rnd}.csv'
    sub_rnd=pd.read_csv(SUB_PATH); sub_rnd[TARGET]=new_preds
    sub_rnd.to_csv(path_rnd,index=False)
    rmse_history.append((f'round-{rnd}', new_rmse))
    preds_history.append((f'round-{rnd}', new_preds))

    if adopted:
        best_preds=new_preds; best_rmse=new_rmse
    print(f'→ Saved {path_rnd}')

print('\n=== RMSE per round ===')
for label, r in rmse_history:
    print(f'  {label}: {r:,.2f}')
print(f'\nBest round: {min(rmse_history,key=lambda x:x[1])[0]}  RMSE={best_rmse:,.2f}')


## 16 · Final Submission

In [ ]:
# Save the best predictions as the primary submission
final_sub=pd.read_csv(SUB_PATH); final_sub[TARGET]=best_preds
final_path=f'{OUT_DIR}/submission_v10_FINAL.csv'
final_sub.to_csv(final_path,index=False)
print(f'Final submission saved → {final_path}')
print(f'Final OOF RMSE: {best_rmse:,.2f}')
print(f'v8 baseline   : 1174.49   delta: {best_rmse-1174.49:+.2f}')


## 17 · Visualisations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, (name, study) in zip(axes.flat, [
    ('XGB sup',  t_xgb_s.study), ('LGB sup',  t_lgb_s.study), ('CAT sup',  t_cat_s.study),
    ('XGB groc', t_xgb_g.study), ('LGB groc', t_lgb_g.study), ('CAT groc', t_cat_g.study),
]):
    vals = [t.value for t in study.trials if t.value is not None]
    ax.plot(vals, alpha=0.4, color='steelblue', lw=0.8)
    ax.plot(pd.Series(vals).cummin(), color='red', lw=1.5, label='best')
    ax.set_title(name); ax.set_xlabel('Trial'); ax.set_ylabel('OOF RMSE (log)')
    ax.legend()
plt.suptitle('v10 Optuna History (n_estimators→5000, lr→0.003)', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# RMSE per pseudo-label round
labels=[l for l,_ in rmse_history]; values=[r for _,r in rmse_history]
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(labels,values,color=['steelblue']+['salmon' if v>=values[0] else 'teal' for v in values[1:]])
ax.axhline(1174.49,color='orange',lw=1.5,ls='--',label='v8 baseline 1174.49')
ax.axhline(1127,color='green',lw=1.5,ls='--',label='target 1127')
for i,(l,v) in enumerate(zip(labels,values)): ax.text(i,v+2,f'{v:.1f}',ha='center',fontsize=9)
ax.set_ylabel('OOF RMSE'); ax.set_title('Pseudo-label round progression'); ax.legend()
plt.tight_layout(); plt.show()


## 18 · Results Summary

In [ ]:
print('='*65)
print('BIGMART v10 — RESULTS SUMMARY')
print('='*65)
print(f'  GBM features           : {len(FEATURES)}')
print(f'  NN OOF RMSE (sup)      : {rmse_nn_sup:.5f} (log)')
print(f'  NN OOF RMSE (groc)     : {rmse_nn_groc:.5f} (log)')
print()
print('  Round-by-round OOF RMSE (raw sales):')
for label,r in rmse_history:
    adopted_str='  ← BEST' if r==best_rmse else ''
    print(f'    {label:10s}: {r:,.2f}{adopted_str}')
print()
print(f'  Final OOF RMSE         : {best_rmse:,.2f}')
print(f'  v8 baseline            : 1174.49   delta: {best_rmse-1174.49:+.2f}')
print(f'  v9 (with leakage fix)  : ~1174     delta: {best_rmse-1174:+.2f}')
print(f'  Target                 : 1127')
print()
print('  v10 changes vs v9:')
print('    + Optuna n_estimators 2000→5000, lr floor 0.005→0.003')
print(f'   + N_TRIALS_SUP={N_TRIALS_SUP}, N_TRIALS_GROC={N_TRIALS_GROC} (was 100)')
print(f'   + Grocery TE alpha={TE_ALPHA_GROC} (was 10) — less shrinkage for sparse items')
print(f'   + {N_PSEUDO_ROUNDS} sequential pseudo-label rounds (was 1, GBM-blend only)')
print('    + Each pseudo round retrains full 4-model stacking pipeline')
print('    + Pseudo rounds adopted only if OOF improves on original train rows')
print()
print(f'  Final submission       : {OUT_DIR}/submission_v10_FINAL.csv')
print('='*65)
